# 7. Teste oficial

Esta é uma etapa independente da busca de hiperparâmetros e da avaliação local. Para usá-la:

1. Copie o arquivo oficial para `data/raw/final_teste.csv`.
2. Abra este notebook e selecione **Run All / Executar tudo**.

O notebook reutiliza os melhores hiperparâmetros registrados em `results/stage4_model_search.json`, ajusta cada pipeline em todos os registros rotulados de `final_treino_val.csv` e calcula scores para os seis algoritmos. Ele não repete as buscas demoradas.

O arquivo de teste pode não ter a coluna `DESFECHO`, como é comum em testes de competição. Nesse caso não é possível calcular AUC nem outras métricas de desempenho: serão gerados scores e classes previstas para inspeção. Se o arquivo tiver `DESFECHO`, esse rótulo será usado somente depois do ajuste para calcular métricas, sem participar de pré-processamento, treinamento, ajuste de hiperparâmetros ou escolha de limiar. Como serão comparados seis modelos, se você escolher um com base nesses resultados rotulados, esse arquivo passa a funcionar como validação e deixa de ser uma avaliação final independente.

Os scores por registro serão gravados em `results/official_test_model_scores.csv`, ignorado pelo Git. Esse arquivo é diagnóstico e inclui resultados dos seis modelos; não é uma submissão pronta, pois o formato exigido pela competição precisa ser conferido separadamente.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
from src.test_evaluation import evaluate_official_test

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "test_evaluation.py").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


print(f"Projeto: {project_root}")
print("Arquivo esperado: data/raw/final_teste.csv")

## Protocolo e proteção contra vazamento

O `id` nunca entra como preditor. Imputação, escala e one-hot encoding são ajustados novamente dentro de cada `Pipeline`, usando apenas o arquivo de treino rotulado. O teste somente recebe `predict_proba` (ou `decision_function` no SVC). Os limiares vêm de scores OOF de desenvolvimento já registrados na etapa 5; nenhum limiar é escolhido olhando os rótulos do teste.

Os hiperparâmetros foram selecionados anteriormente no conjunto de desenvolvimento da etapa 4 e são reutilizados aqui. Para impedir aplicação acidental de parâmetros obsoletos, a rotina confere o número de linhas e o hash SHA-256 do CSV rotulado contra os metadados da busca. Se o arquivo rotulado mudar, atualize as etapas de busca/limiar antes de continuar.

In [ ]:
metrics_table, prediction_scores, roc_figure, test_summary = evaluate_official_test(
    project_root=project_root,
    train_filename="final_treino_val.csv",
    test_filename="final_teste.csv",
)

if test_summary["test_labels_available"]:
    print("Métricas calculadas com os rótulos presentes no arquivo de teste.")
    display(metrics_table)
    if roc_figure is not None:
        display(roc_figure)
else:
    print("O teste não contém DESFECHO: métricas de desempenho não podem ser calculadas.")
    print("Scores dos seis modelos (sem rótulo verdadeiro):")
    display(metrics_table[["Modelo", "Tipo de score", "Limiar (OOF treino)"]])

print(f"Registros pontuados: {test_summary['test_rows_scored']:,}")
print("Métricas agregadas: results/official_test_metrics.json")
print("Scores por registro: results/official_test_model_scores.csv (local/ignorado pelo Git)")

## Como interpretar a saída

A AUC-ROC, quando calculável, é a comparação principal e usa scores contínuos; o score do SVC é `decision_function`, não probabilidade calibrada. Sensibilidade, especificidade, LR− e F1 usam limiares definidos no desenvolvimento e são secundários.